In [0]:
# Healthcare Silver Transformation

from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalog_schema = "workspace.healthcare"

print("Silver Transformation Started")

In [0]:
patients_df = spark.table(f"{catalog_schema}.bronze_patients")
doctors_df = spark.table(f"{catalog_schema}.bronze_doctors")
appointments_df = spark.table(f"{catalog_schema}.bronze_appointments")
billing_df = spark.table(f"{catalog_schema}.bronze_billing")
print("All Bronze tables loaded susscefully")

In [0]:
# Remove duplicates and invalid patient records

silver_patients = (
    patients_df
    .dropDuplicates(["PatientID"])
    .filter(F.col("PatientID").isNotNull())
    .filter(F.col("PatientName").isNotNull())
    .filter(F.col("Age").between(0, 120))
    .withColumn("PatientName", F.trim(F.col("PatientName")))
    .withColumn(
        "AgeGroup",
        F.when(F.col("Age") < 18, "Child")
         .when(F.col("Age") < 60, "Adult")
         .otherwise("Senior")
    )
)

display(silver_patients)

In [0]:
# Remove duplicates and invalid doctor records

silver_doctors = (
    doctors_df
    .dropDuplicates(["DoctorID"])
    .filter(F.col("DoctorID").isNotNull())
    .filter(F.col("DoctorName").isNotNull())
    .filter(F.col("Department").isNotNull())
    .filter(F.col("Experience") >= 0)
    .withColumn("DoctorName", F.trim(F.col("DoctorName")))
    .withColumn(
        "ExperienceLevel",
        F.when(F.col("Experience") < 5, "Junior")
         .when(F.col("Experience") < 10, "Mid-Level")
         .otherwise("Senior")
    )
)

display(silver_doctors)

In [0]:
# Standardize dates and validate appointments

silver_appointments = (
    appointments_df
    .dropDuplicates(["AppointmentID"])
    .filter(F.col("AppointmentID").isNotNull())
    .filter(F.col("PatientID").isNotNull())
    .filter(F.col("DoctorID").isNotNull())
    .withColumn(
        "AppointmentDate",
        F.to_date(F.col("AppointmentDate"))
    )
    .filter(F.col("AppointmentDate").isNotNull())
    .filter(
        F.col("Status").isin(
            "Completed", "Pending", "Cancelled"
        )
    )
)

# Keep only appointments with valid patient and doctor references
silver_appointments = (
    silver_appointments
    .join(
        silver_patients.select("PatientID"),
        on="PatientID",
        how="inner"
    )
    .join(
        silver_doctors.select("DoctorID"),
        on="DoctorID",
        how="inner"
    )
)

display(silver_appointments)

In [0]:
# Clean billing records

silver_billing = (
    billing_df
    .dropDuplicates(["BillingID"])
    .filter(F.col("BillingID").isNotNull())
    .filter(F.col("AppointmentID").isNotNull())
    .filter(F.col("Amount") >= 0)
    .filter(
        F.col("PaymentStatus").isin(
            "Paid", "Pending", "Cancelled"
        )
    )
)

# Keep billing records linked to valid appointments
silver_billing = (
    silver_billing
    .join(
        silver_appointments.select("AppointmentID"),
        on="AppointmentID",
        how="inner"
    )
)

display(silver_billing)

In [0]:
# Save transformed datasets into Silver layer

silver_tables = {
    "silver_patients": silver_patients,
    "silver_doctors": silver_doctors,
    "silver_appointments": silver_appointments,
    "silver_billing": silver_billing
}

for table_name, df in silver_tables.items():

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{catalog_schema}.{table_name}")
    )

    print(f"Successfully saved: {table_name}")

print("Silver transformation completed!")

In [0]:
# Verify all Silver tables

for table_name in silver_tables.keys():

    df = spark.table(f"{catalog_schema}.{table_name}")

    print(f"\nTable: {table_name}")
    print(f"Total Records: {df.count()}")

    df.printSchema()
    df.show(5, truncate=False)